In [1]:
import pickle
import pandas as pd
import math
import requests

In [2]:
try:
    hist = pd.read_csv('processed_history.csv')
    hist['Date'] = pd.to_datetime(hist['Date'])
except:
    hist = pd.DataFrame()

attack = pickle.load(open('attack.pkl','rb'))
defence = pickle.load(open('defence.pkl','rb'))
league = pickle.load(open('league_avg.pkl','rb'))
LEAGUE_HOME = league['home_avg']; LEAGUE_AWAY = league['away_avg']

STADIUMS = {
    'Arsenal':(51.555,-0.108),'Chelsea':(51.4816,-0.191),'Man City':(53.483,-2.200),'Man United':(53.463,-2.291),
    'Liverpool':(53.4308,-2.9608),'Tottenham':(51.6043,-0.0664),'Newcastle':(54.9756,-1.6217),'Everton':(53.4388,-2.966),
    'West Ham':(51.538,-0.016),'Brighton':(50.861,-0.083),'Aston Villa':(52.509,-1.884),'Brentford':(51.4907,-0.288),
    'Fulham':(51.4749,-0.2216),'Crystal Palace':(51.3983,-0.0855),'Nottm Forest':(52.9399,-1.2108),'Wolves':(52.5902,-2.1304),
    'Leeds':(53.7778,-1.5722),'Sunderland':(54.9146,-1.3884),'Burnley':(53.789,-2.2302),'Bournemouth':(50.7352,-1.8383)
}

def poisson(lam, k): return (math.exp(-lam) * (lam**k)) / math.factorial(k)

def get_rest_days(team, new_date_str, manual_rest=None):
    if manual_rest is not None: return manual_rest
    new_date = pd.to_datetime(new_date_str)
    if hist.empty: return 7
    tg = hist[((hist['HomeTeam']==team) | (hist['AwayTeam']==team)) & (hist['Date'] < new_date)]
    if tg.empty: return 7
    last = tg.sort_values('Date').iloc[-1]
    return (new_date - last['Date']).days

def get_weather(team, date):
    lat, lon = STADIUMS.get(team,(51.5,-0.12))
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=precipitation,wind_speed_10m,temperature_2m&start_date={date}&end_date={date}"
    try:
        r = requests.get(url, timeout=10).json()
        rain = sum(r['hourly']['precipitation']); wind = max(r['hourly']['wind_speed_10m']); temp = sum(r['hourly']['temperature_2m'])/len(r['hourly']['temperature_2m'])
        print(f"Weather {team}: Rain {rain:.1f}mm, Wind {wind:.1f} km/h, Temp {temp:.1f}C")
        return rain, wind, temp
    except: return 0,10,15

def predict_v9_complete(home, away, date="2026-11-22", home_odds=2.0, draw_odds=3.4, away_odds=3.8, under_odds=2.0, over_odds=1.85, manual_home_rest=None, manual_away_rest=None):
    ha = sum(attack[home][-10:])/10; hd = sum(defence[home][-10:])/10
    aa = sum(attack[away][-10:])/10; ad = sum(defence[away][-10:])/10
    ph = (ha / LEAGUE_HOME) * (ad / LEAGUE_AWAY) * LEAGUE_HOME
    pa = (aa / LEAGUE_AWAY) * (hd / LEAGUE_HOME) * LEAGUE_AWAY
    ph = 0.8*ph + 0.2*LEAGUE_HOME; pa = 0.8*pa + 0.2*LEAGUE_AWAY
    ph = max(0.3, min(3.0, ph)); pa = max(0.2, min(2.6, pa))

    home_rest = get_rest_days(home, date, manual_home_rest); away_rest = get_rest_days(away, date, manual_away_rest)
    rain, wind, temp = get_weather(home, date)
    penalty = 0
    if rain > 5: penalty += 4
    if wind > 40: penalty += 3
    if temp < 2: penalty += 2
    if home_rest < 3: penalty += 2
    if away_rest < 3: penalty += 1

    # Probs
    p0 = poisson(ph,0)*poisson(pa,0)
    p1 = poisson(ph,1)*poisson(pa,0) + poisson(ph,0)*poisson(pa,1)
    p2 = poisson(ph,2)*poisson(pa,0) + poisson(ph,1)*poisson(pa,1) + poisson(ph,0)*poisson(pa,2)
    over_prob = (1 - p0 - p1 - p2)*100
    over_adj = max(5, min(95, over_prob - penalty)); under_prob = 100 - over_adj

    p_home = sum(poisson(ph,h)*poisson(pa,a) for h in range(6) for a in range(6) if h>a)
    p_draw = sum(poisson(ph,h)*poisson(pa,a) for h in range(6) for a in range(6) if h==a)
    p_away = sum(poisson(ph,h)*poisson(pa,a) for h in range(6) for a in range(6) if h<a)
    tot = p_home+p_draw+p_away; p_home,p_draw,p_away = p_home/tot*100, p_draw/tot*100, p_away/tot*100

    scores = {}
    for h in range(0,5):
        for a in range(0,5):
            scores[f"{h}-{a}"] = poisson(ph,h)*poisson(pa,a)*100
    top_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]

    print(f"\n=== {home} vs {away} | {date} | V9 COMPLETE | 90 MINS ===")
    print(f"Expected: {home} {ph:.2f} - {pa:.2f} {away} | Rest {home_rest} vs {away_rest} days | Penalty -{penalty}% on Over")
    print(f"\n--- 1. WIN (1X2) ---")
    print(f"Home Win {p_home:.1f}% (Bookie {1/home_odds*100:.1f}%) | Draw {p_draw:.1f}% (Bookie {1/draw_odds*100:.1f}%) | Away Win {p_away:.1f}% (Bookie {1/away_odds*100:.1f}%)")
    # Value check for Win
    for name, prob, odds in [('Home',p_home,home_odds),('Draw',p_draw,draw_odds),('Away',p_away,away_odds)]:
        bookie = 1/odds*100
        if prob > bookie + 8: print(f"  ✅ VALUE {name} - Model {prob:.1f}% vs Bookie {bookie:.1f}% Edge +{prob-bookie:.1f}%")
    print(f"\n--- 2. OVER / UNDER ---")
    print(f"Over 2.5: {over_adj:.1f}% (Bookie {1/over_odds*100:.1f}%) | Under 2.5: {under_prob:.1f}% (Bookie {1/under_odds*100:.1f}%)")
    if over_adj < 40 and under_prob > (1/under_odds*100 + 8):
        print(f"  ✅ BET UNDER 2.5 - Edge +{under_prob - (1/under_odds*100):.1f}%")
    elif over_adj < 45:
        print(f"  ⚠️ Lean Under - No strong bet")
    else:
        print(f"  ❌ SKIP Over/Under - No edge")
    print(f"\n--- 3. CORRECT SCORE ---")
    for s,p in top_scores: print(f"  {s} : {p:.1f}%")
    print(f"Most Likely: {top_scores[0][0]} ({top_scores[0][1]:.1f}%) | Top 3 Cover: {sum([p for _,p in top_scores[:3]]):.1f}%")
    print(f"Note: Extra Time NO count. 90+ injury DOES count.")
    return top_scores

if __name__ == "__main__":
    predict_v9_complete("Arsenal","Chelsea", date="2026-11-22", home_odds=1.90, draw_odds=3.60, away_odds=4.20, under_odds=1.95, over_odds=1.85)

Weather Arsenal: Rain 0.0mm, Wind 19.8 km/h, Temp 6.6C

=== Arsenal vs Chelsea | 2026-11-22 | V9 COMPLETE | 90 MINS ===
Expected: Arsenal 1.91 - 0.74 Chelsea | Rest 15 vs 14 days | Penalty -0% on Over

--- 1. WIN (1X2) ---
Home Win 64.6% (Bookie 52.6%) | Draw 21.5% (Bookie 27.8%) | Away Win 13.9% (Bookie 23.8%)
  ✅ VALUE Home - Model 64.6% vs Bookie 52.6% Edge +12.0%

--- 2. OVER / UNDER ---
Over 2.5: 49.2% (Bookie 54.1%) | Under 2.5: 50.8% (Bookie 51.3%)
  ❌ SKIP Over/Under - No edge

--- 3. CORRECT SCORE ---
  1-0 : 13.6%
  2-0 : 12.9%
  1-1 : 10.0%
  2-1 : 9.5%
  3-0 : 8.2%
Most Likely: 1-0 (13.6%) | Top 3 Cover: 36.5%
Note: Extra Time NO count. 90+ injury DOES count.
